# OC Transpo Optimizer — Equation Formulation

This notebook is an intermediate step: it **writes down the model equations** from the current final report draft in a clean order before the next solver iteration.

The formulation below follows the current bus-allocation version of the model, where the main decision variable is the **integer number of buses assigned to each route and time block**.


#### 1. Imports

We only need light imports here, mainly so later iterations can extend this notebook into a solver notebook.


In [1]:
import numpy as np
from IPython.display import display, Markdown

# This notebook is equation-first.
# Numerical data / solver calls will be added in a later iteration.


#### 2. Sets, indices, and decision variables

Let

- $R$ = set of routes, indexed by $r \in R$
- $T$ = set of time blocks, indexed by $t \in T$

Time blocks currently used in the report:

- Early AM
- AM Peak
- Midday
- PM Peak
- Evening
- Night

Primary decision variable:

$$
n_{new,r,t} \in \mathbb{Z}_{\ge 0}
$$

where $n_{new,r,t}$ is the number of buses assigned to route $r$ during time block $t$.

Baseline / known quantities include:

- $n_{old,r,t}$ = current number of buses
- $x_{old,r,t}$ = baseline ridership
- $L_r$ = route length
- $T_{r,t}$ = average route cycle / round-trip time
- $H_{block,t}$ = service hours in time block $t$


#### 3. Frequency relationships

Frequency is buses per hour. The report uses the relationship

$$
f_{r,t} = \frac{n_{r,t}}{T_{r,t}}
$$

so that baseline and new frequencies are

$$
f_{old,r,t} = \frac{n_{old,r,t}}{T_{r,t}},
\qquad
f_{new,r,t} = \frac{n_{new,r,t}}{T_{r,t}}.
$$

The earlier frequency-based baseline relationship also appears as

$$
f_{old,r,t} = \frac{60}{\bar{h}_{r,t}},
$$

where $\bar{h}_{r,t}$ is the average scheduled headway in minutes.


In [2]:
def frequency_from_buses(n, T_rt):
    return n / T_rt

def baseline_frequency_from_headway(headway_min):
    return 60.0 / headway_min


#### 4. Demand / ridership model

The demand model uses elasticity with respect to service frequency. In the project materials, elasticity is taken as $\epsilon = 0.5$.

The earlier frequency-based form is written as

$$
x_{new,r,t}(f) = x_{old,r,t}\left(\frac{f_{new,r,t}}{f_{old,r,t}}\right)^{\epsilon}
$$

and in the intermediate project writeup this appears with $\epsilon = 0.5$ as

$$
x_{new,r,t}(f) = x_{old,r,t}\left(\frac{f_{old,r,t}}{f_{new,r,t}}\right)^{0.5}.
$$

Using $f_{r,t} = n_{r,t}/T_{r,t}$, the bus-count form written in the final report draft is

$$
x_{new,r,t}(n)
=
x_{old,r,t}
\left(
\frac{n_{old,r,t}/T_{r,t}}{n_{new,r,t}/T_{r,t}}
\right)^{0.5}
=
x_{old,r,t}
\left(
\frac{n_{old,r,t}}{n_{new,r,t}}
\right)^{0.5}.
$$

For this notebook, I am recording the equations **as written in the draft** so the next iteration can sanity-check the final sign / direction convention before solving.


In [3]:
def demand_from_frequency(f_new, x_old, f_old, elasticity=0.5):
    f_new = np.maximum(f_new, 1e-9)
    f_old = np.maximum(f_old, 1e-9)
    # Recorded as written in the draft / current notes
    return x_old * (f_old / f_new) ** elasticity

def demand_from_buses(n_new, x_old, n_old, elasticity=0.5):
    n_new = np.maximum(n_new, 1e-9)
    n_old = np.maximum(n_old, 1e-9)
    # Recorded as written in the draft / current notes
    return x_old * (n_old / n_new) ** elasticity


#### 5. Operating cost equations

### 5.1 Labour cost change

$$
\Delta C_{labour,r,t}
=
\left(n_{new,r,t} - n_{old,r,t}\right)\, H_{block,t}\, W_{driver}
$$

where $W_{driver}$ is the hourly driver wage.

### 5.2 Fuel cost change

$$
\Delta C_{fuel,r,t}
=
\left(n_{new,r,t} - n_{old,r,t}\right)
\, L_r \,
\frac{H_{block,t}}{T_{r,t}}
\, P_{fuel}\, FC
$$

where

- $P_{fuel}$ = fuel price
- $FC$ = fuel consumption factor

### 5.3 Maintenance cost change

$$
\Delta C_{maintenance,r,t}
=
\left(n_{new,r,t} - n_{old,r,t}\right)
\, L_r \,
\frac{H_{block,t}}{T_{r,t}}
\, P_{maintenance}
$$

where $P_{maintenance}$ is the maintenance cost per km traveled.


In [4]:
def delta_labour_cost(n_new, n_old, H_block_t, W_driver):
    return (n_new - n_old) * H_block_t * W_driver

def delta_fuel_cost(n_new, n_old, L_r, H_block_t, T_rt, P_fuel, FC):
    return (n_new - n_old) * L_r * (H_block_t / T_rt) * P_fuel * FC

def delta_maintenance_cost(n_new, n_old, L_r, H_block_t, T_rt, P_maintenance):
    return (n_new - n_old) * L_r * (H_block_t / T_rt) * P_maintenance


#### 6. Benefit equations

6.1 Waiting-time benefit

Average wait time is taken as half the headway, so the total waiting-time benefit is

$$
\Delta B_{time,r,t}
=
\frac{1}{2}
\left(
\frac{1}{f_{old,r,t}} - \frac{1}{f_{new,r,t}}
\right)
x_{new,r,t}
\, W_{passenger}\, F_{wage,t}.
$$

Substituting $f_{r,t} = n_{r,t}/T_{r,t}$ gives the draft form

$$
\Delta B_{time,r,t}
=
\frac{T_{r,t}}{2}
\left(
\frac{1}{n_{old,r,t}} - \frac{1}{n_{new,r,t}}
\right)
x_{old,r,t}
\left(
\frac{n_{old,r,t}}{n_{new,r,t}}
\right)^{0.5}
W_{passenger}\, F_{wage,t}.
$$

6.2 Emissions benefit

Avoided-emissions benefit given as

$$
\Delta B_{emissions,r,t}
=
IGHG_{car,t}\, Vkm_{saved}\, (x_{new,r,t} - x_{old,r,t})\, L_{r,passenger}\, SCC
$$

and after substitution,

$$
\Delta B_{emissions,r,t}
=
IGHG_{car,t}\, Vkm_{saved}
\left[
x_{old,r,t}
\left(
\frac{n_{old,r,t}}{n_{new,r,t}}
\right)^{0.5}
- x_{old,r,t}
\right]
L_{r,passenger}\, SCC.
$$

A simple route-level approximation is proposed: 

$$
L_{r,passenger} \approx 0.4\, L_r.
$$


In [5]:
def delta_wait_time_benefit_from_frequency(f_new, f_old, x_new, W_passenger, F_wage_t):
    f_new = np.maximum(f_new, 1e-9)
    f_old = np.maximum(f_old, 1e-9)
    return 0.5 * ((1.0 / f_old) - (1.0 / f_new)) * x_new * W_passenger * F_wage_t

def delta_wait_time_benefit_from_buses(n_new, n_old, T_rt, x_old, W_passenger, F_wage_t, elasticity=0.5):
    n_new = np.maximum(n_new, 1e-9)
    n_old = np.maximum(n_old, 1e-9)
    x_new = demand_from_buses(n_new, x_old, n_old, elasticity=elasticity)
    return (T_rt / 2.0) * ((1.0 / n_old) - (1.0 / n_new)) * x_new * W_passenger * F_wage_t

def delta_emissions_benefit(n_new, n_old, x_old, IGHG_car_t, Vkm_saved, L_passenger_r, SCC, elasticity=0.5):
    x_new = demand_from_buses(n_new, x_old, n_old, elasticity=elasticity)
    return IGHG_car_t * Vkm_saved * (x_new - x_old) * L_passenger_r * SCC


#### 7. Net objective function

The current draft frames the model as minimizing net cost after subtracting monetized benefits:

$$
\min_{n_{new,r,t}}
\sum_{r\in R}\sum_{t\in T}
\left(
\Delta C_{labour,r,t}
+
\Delta C_{maintenance,r,t}
+
\Delta C_{fuel,r,t}
-
\Delta B_{time,r,t}
-
\Delta B_{emissions,r,t}
\right).
$$

Equivalently, this could later be written as a maximization of net benefit. For now, I am keeping the report's minimization form.


#### 8. Constraints

8.1 Total operating-budget constraint

$$
\sum_{r\in R}\sum_{t\in T}
\left(
\Delta C_{labour,r,t}
+
\Delta C_{maintenance,r,t}
+
\Delta C_{fuel,r,t}
\right)
\le
Budget_{TotalOperational}
$$

8.2 Fleet-availability constraint by time block

$$
\sum_{r\in R} n_{new,r,t} \le Buses_{tot},
\qquad \forall t \in T
$$

8.3 Route/time lower and upper bounds

$$
n^{min}_{r,t} \le n_{new,r,t} \le n^{max}_{r,t},
\qquad \forall r \in R,\; \forall t \in T
$$

8.4 Integrality

$$
n_{new,r,t} \in \mathbb{Z}_{\ge 0},
\qquad \forall r \in R,\; \forall t \in T
$$


#### 9. Compact formulation

Putting everything together:

$$
\begin{aligned}
\min_{n_{new,r,t}} \quad &
\sum_{r\in R}\sum_{t\in T}
\left(
\Delta C_{labour,r,t}
+
\Delta C_{maintenance,r,t}
+
\Delta C_{fuel,r,t}
-
\Delta B_{time,r,t}
-
\Delta B_{emissions,r,t}
\right) \\
\text{s.t.}\quad &
\sum_{r\in R}\sum_{t\in T}
\left(
\Delta C_{labour,r,t}
+
\Delta C_{maintenance,r,t}
+
\Delta C_{fuel,r,t}
\right)
\le Budget_{TotalOperational}, \\
&
\sum_{r\in R} n_{new,r,t} \le Buses_{tot},
\qquad \forall t \in T, \\
&
n^{min}_{r,t} \le n_{new,r,t} \le n^{max}_{r,t},
\qquad \forall r \in R,\; \forall t \in T, \\
&
n_{new,r,t} \in \mathbb{Z}_{\ge 0},
\qquad \forall r \in R,\; \forall t \in T.
\end{aligned}
$$


#### 10. Issues Log

- current bus-count formulation is a mixted integer non-linear model because the demand and benefit terms depend nonlinearly on $n_{new,r,t}$.
- sign convention should be checked carefully in the implementation so that "more buses" actually increases ridership and benefits in the intended direction.
- Once the baseline dataset is assembled, next notebook can decide whether to:
   - solve the nonlinear model directly,
   - linearize / approximate parts of it,
   - or build a MILP using piecewise-linear approximations.
